In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import skimage.io
import time
import torch
import glob
from scipy.spatial import KDTree
from scipy.spatial import Delaunay

import sys
sys.path.append("../src")
from geometry import *

X = get_image_curve("../data/images/fish.png")
#t = np.linspace(0, 2*np.pi, 1000)
#X = np.array([np.cos(t), np.sin(t)]).T
X = X-np.min(X, axis=0, keepdims=True)
X = X/np.max(np.abs(X))

N = estimate_normals_curve(X)

In [ ]:
plt.scatter(X[:, 0], X[:, 1], s=5)

In [ ]:
res = 128
anchors = np.meshgrid(np.linspace(1.2, -0.2, res), np.linspace(-0.2, 1.2, res), indexing='ij')
anchors = np.concatenate(tuple([Ai.flatten()[:, None] for Ai in anchors]), axis=1)

tree = KDTree(X)
d, _ = tree.query(anchors, k=1, workers=-1)
d = d.flatten()
anchors = anchors[d < 0.05, :]

sdf = get_sdf(X, N, anchors)
sdf = np.sign(sdf)
sdf = 0.5*(1+sdf)
print(anchors.shape, X.shape)
anchors = np.concatenate((anchors, X), axis=0)
sdf = np.concatenate((sdf, 0.5*np.ones(X.shape[0])))

#plt.scatter(X[:, 0], X[:, 1])
plt.scatter(anchors[:, 0], anchors[:, 1], c=sdf, cmap='RdBu')
plt.colorbar()

In [ ]:
# Number of points in each set
n_points = 50
idx = np.random.randint(0, X.shape[0], n_points)
XIn = X[idx, :] - 0.005*N[idx, :]
idx = np.random.randint(0, X.shape[0], n_points)
XOut = X[idx, :] + 0.005*N[idx, :]

n_iters = 500
add_every = 10
n_extra = 2*n_iters//add_every # Add two new points every so often
XExtra = np.zeros((n_extra, 2))

XSamples = np.concatenate((XIn, XOut, XExtra), axis=0)

In [ ]:
anchors = torch.from_numpy(np.array(anchors))
sdf = torch.from_numpy(np.array(sdf))
XSamples  = torch.from_numpy(XSamples).requires_grad_()
c = torch.zeros(n_points*2 + n_extra)
c[n_points:2*n_points] = 1
c[n_points*2::2] = 1 # Every other extra point is a 0 or 1 to keep things even

In [ ]:
res = 512
G = np.meshgrid(np.linspace(1.2, -0.2, res), np.linspace(-0.2, 1.2, res), indexing='ij')
G = np.concatenate(tuple([Gi.flatten()[:, None] for Gi in G]), axis=1)


optimizer = torch.optim.Adam([XSamples], lr=3e-4)

temperature = 1
losses = []
pidx = 0

fac = 2
plt.figure(figsize=(10*fac, 10*fac))
n_samples_using = n_points*2
for it in range(n_iters):
    XSUsing = XSamples.detach()[0:n_samples_using, :].cpu().numpy()
    optimizer.zero_grad()

    ## Compute distance from all pixels to landmarks
    n_neighbs = 2
    tree = KDTree(XSUsing)
    _, nn_idx = tree.query(anchors.detach().cpu().numpy(), k=n_neighbs, workers=-1) # Nearest landmark indices to anchors
    
    ## Based on the landmark locations, compute the weight influence
    ## of each landmark on each pixel using softmax
    weights = torch.zeros(anchors.shape[0], n_neighbs)
    for i in range(n_neighbs):
        dX = XSamples[nn_idx[:, i], :] - anchors
        weights[:, i] = 1/(1e-4+torch.sum(dX**2, dim=1))
    mx = weights[:, 0].view(weights.shape[0], 1)
    weights = torch.exp(temperature*weights/mx)
    weights = weights/torch.sum(weights, dim=1, keepdims=True)

    ## Splat colors onto anchors
    J = torch.zeros(anchors.shape[0])
    for i in range(n_neighbs):
        J += c[nn_idx[:, i]]*weights[:, i]
    all_losses = (J-sdf)**2

    # Compute loss
    loss = torch.sum(all_losses)
    losses.append(loss.item())
    
    loss.backward()
    optimizer.step()
    XSUsing = XSamples.detach()[0:n_samples_using, :].cpu().numpy()
    if it%add_every == 0:
        # Figure out which point on the curve (the last X.shape[0] indices) has the biggest loss and add two sites there
        max_err_idx = torch.argmax(all_losses[-X.shape[0]:]) + all_losses.shape[0]-X.shape[0]
        with torch.no_grad():
            XSamples[n_samples_using:n_samples_using+2, :] = anchors[max_err_idx, :]
        n_samples_using += 2
    
    if it%5 == 0:
        plt.clf()
        plt.subplot(221)
        plt.plot(X[:, 0], X[:, 1], linewidth=1, c='k')
        plt.scatter(XSUsing[:, 0], XSUsing[:, 1], c=c.numpy()[0:XSUsing.shape[0]], s=1, cmap='RdBu')
        if it%add_every == 0:
            plt.scatter(anchors[max_err_idx, 0], anchors[max_err_idx, 1])
        plt.axis("equal")
        plt.xlim([-0.2, 1.2])
        plt.ylim([-0.2, 1.2])
        plt.title("Using {} sites".format(n_samples_using))
    
        plt.subplot(222)
        tree = KDTree(XSUsing)
        _, idx = tree.query(G)
        idx = c[idx].detach().cpu().numpy()
        plt.imshow(np.fliplr(idx.reshape((res, res)).T), cmap='RdBu', extent=(-0.2, 1.2, -0.2, 1.2), origin='lower')
        if it%add_every == 0:
            plt.scatter(anchors[max_err_idx, 0], anchors[max_err_idx, 1])
            plt.title("Adding two sites")
    
        ## Plot reconstruction so far
        plt.subplot(223)
        tri = Delaunay(XSUsing)
        simplices = tri.simplices
        centers = get_circumcenters(XSUsing, simplices)
        # Need to be able to quickly look up two involved triangles for an edge
        edge2tris = {}
        for i, simplex in enumerate(simplices):
            for j in range(len(simplex)):
                edge = [simplex[j], simplex[(j+1)%len(simplex)]]
                edge = tuple(sorted(edge))
                if not edge in edge2tris:
                    edge2tris[edge] = []
                edge2tris[edge].append(i)
        plt.plot(X[:, 0], X[:, 1], c='k', linestyle='--')
        for (e1, e2), tris in edge2tris.items():
            # Only plot edges between inside/outside, as well as non-boundary edges
            if c[e1] != c[e2] and len(tris) == 2:
                tri1, tri2 = tris
                Xi = centers[[tri1, tri2], :]
                plt.plot(Xi[:, 0], Xi[:, 1])
        
        
        plt.subplot(224)
        plt.plot(losses)
        plt.title("Loss {:.5f}".format(loss.item()))

        
        plt.savefig("Iter{}.png".format(pidx))
        pidx += 1

